# ☢️ Interactive Nuclear Reactor Kinetics Simulator

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
from google.colab import output
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

def reactor_equations(t, y, Lambda, beta, lam, rho):
    n, C = y[0], y[1]
    dn_dt = ((rho - beta) / Lambda) * n + lam * C
    dC_dt = (beta / Lambda) * n - lam * C
    return [dn_dt, dC_dt]

def run_live_simulation(change=None):
    with output_box:
        clear_output(wait=True)
        is_scram = scram_toggle.value
        if is_scram:
            rho_inserted = -0.05
            print("🚨 EMERGENCY SCRAM ACTIVATED: Control Rods Fully Inserted!")
        else:
            rho_inserted = rod_slider.value
            print(f"🕹️ Control Room Status: Rod Reactivity set to {rho_inserted:.4f}")
            
        beta = beta_slider.value
        Lambda, lam = 1e-4, 0.08
        
        t_eval = np.linspace(0, 15, 1000)
        initial_state = [1.0, (beta / (Lambda * lam)) * 1.0]
        solution = solve_ivp(reactor_equations, (0.0, 15.0), initial_state, args=(Lambda, beta, lam, rho_inserted), t_eval=t_eval)
        power_level = solution.y[0]
        
        plt.figure(figsize=(10, 4.5))
        plt.plot(solution.t, power_level, color="green" if rho_inserted <= 0 else "red", linewidth=2.5, label="Power Level (n)")
        plt.title("☢️ Live Nuclear Reactor Core Response Timeline", fontsize=13, fontweight='bold')
        plt.xlabel("Time (Seconds)")
        plt.ylabel("Relative Power Level (1.0 = 100%)")
        plt.grid(True, linestyle="--", alpha=0.5)
        plt.ylim(0, max(2.1, max(power_level) * 1.1))
        plt.legend(loc="upper left")
        plt.show()
        print(f"📈 Peak Power Reached: {max(power_level):.2f}x of baseline operating capacity.")

rod_slider = widgets.FloatSlider(value=0.0015, min=-0.0050, max=0.0050, step=0.0005, description='Control Rods:', layout=widgets.Layout(width='450px'))
beta_slider = widgets.FloatSlider(value=0.0065, min=0.0040, max=0.0080, step=0.0005, description='Beta Fraction:', layout=widgets.Layout(width='450px'))
scram_toggle = widgets.ToggleButton(value=False, description='💥 TRIGGER EMERGENCY SCRAM', button_style='danger', layout=widgets.Layout(width='250px', height='35px'))
output_box = widgets.Output()

rod_slider.observe(run_live_simulation, names='value')
beta_slider.observe(run_live_simulation, names='value')
scram_toggle.observe(run_live_simulation, names='value')

control_panel = widgets.VBox([widgets.HTML("<h3>⚙️ NUCLEAR REACTOR CONTROL CONSOLE</h3>"), rod_slider, beta_slider, scram_toggle])
display(widgets.VBox([control_panel, output_box]))
run_live_simulation()